# lif saver

Loads one channel of one scene from a raw `.lif` file (via `aicsimageio`, so this notebook
must run in `leica-env`) and saves it as a plain, voxel-calibrated ImageJ TIFF.

Exists because `micro-sam-env` (the dedicated environment created for the interactive SAM
annotator, see `26_07_30_Lisa_MB_handlabel_sam.ipynb`) deliberately does not have
`aicsimageio` installed -- it still needs `tifffile <2023.3.15` to satisfy `aicsimageio`'s
own pin, which is incompatible with the newer `tifffile`/`napari`/`vispy` combination
`micro-sam-env` needs to avoid the OpenGL rendering crash found in `leica-env`. So `.lif`
loading happens here, once, in `leica-env`; downstream notebooks running in
`micro-sam-env` just read the plain TIFF this produces with bare `tifffile`, no
`aicsimageio`/`data_processing` import needed at all.

In [1]:
import os
import sys
import platform
import numpy as np
from aicsimageio import AICSImage
from tifffile import imwrite

system = platform.system()
if system == 'Linux':
    home = '/home/gerard/'
elif system == 'Darwin':
    home = '/Users/gerard/'
elif system == 'Windows':
    home = 'C:/Users/cviko/'

try:
    sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
    from data_processing import describe_acquisition
except ImportError:
    path2add = home + 'analysis/confocal/src'
    sys.path.append(path2add)
    from data_processing import describe_acquisition

data_home = home + 'data/confocal/'

02-Sep-26 16:23:29 - bfio.backends - WARNING  - Java backend is not available. This could be due to a missing dependency (jpype).
/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


## Parameters -- edit per scene/channel you want to save

In [2]:
date = '2026_07_30'
user = 'Lisa'
scene = 0
channel = 0  # ch0 = nc82

lif_path = data_home + date + '_' + user + '/Project.lif'

info = describe_acquisition(lif_path, do_print=False)
scene_names = list(info.keys())
print('scenes:', scene_names)
print('saving scene', scene, f'({scene_names[scene]})', 'channel', channel)

scenes: ['NGS_20min', 'Rotiblock_20min', 'NGS_40min', 'RotiBlock_40min']
saving scene 0 (NGS_20min) channel 0


## Load from `.lif` and save as a plain calibrated TIFF

In [3]:
img = AICSImage(lif_path)
img.set_scene(img.scenes[scene])

vxy = info[scene_names[scene]]['voxel_xy_um']
vz = info[scene_names[scene]]['voxel_z_um']
print(f'scene {scene} ({scene_names[scene]}): vxy={vxy:.4f} um/px, vz={vz:.4f} um/step')

stack = img.get_image_data('ZYX', T=0, C=channel).astype(np.float32)
print('stack shape (ZYX):', stack.shape)

out_dir = data_home + date + '_' + user + f'/series_{scene}/'
os.makedirs(out_dir, exist_ok=True)
out_path = out_dir + f'{date}_s{scene}_ch{channel}.tif'

imwrite(out_path, stack, imagej=True, resolution=(1 / vxy, 1 / vxy),
        metadata={'spacing': vz, 'unit': 'um', 'axes': 'ZYX'})
print('saved to', out_path)

scene 0 (NGS_20min): vxy=0.2290 um/px, vz=0.9880 um/step
stack shape (ZYX): (86, 1024, 1024)
saved to /home/gerard/data/confocal/2026_07_30_Lisa/series_0/2026_07_30_s0_ch0.tif
